# Debugging PyTorch: the errors everyone hits, and a method to find them

_PyTorch_

**Most of your PyTorch time is debugging — so learn the greatest-hits errors, their fixes, and one method: overfit a single batch first.**

PyTorch errors feel random until you see they come from a small, fixed set of mismatches:
       shapes that do not line up, devices that disagree, dtypes that are wrong, or
       gradients that are missing, exploding, or never zeroed. Almost every message maps to one of these.

       So the cure is not memorizing messages &mdash; it is a routine that exposes the mismatch:
       set a seed (so the bug is reproducible), overfit a single batch (separates wiring bugs from data
       bugs), print shapes, dtypes, and devices at the failing line, and check gradient norms (zero means
       no learning signal; huge means it is about to explode). When the routine still hides the source, turn on
       set_detect_anomaly(True) and let PyTorch point at the operation.

---

This notebook walks the method step by step — seed, overfit one batch, print shapes/dtypes/devices, check gradient norms, then catch the silent broadcasting and in-place bugs. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Set a seed, then reproduce a shape mismatch and fix it

Step one of the method is `torch.manual_seed(0)` so the bug is reproducible. The most common error — `mat1 and mat2 shapes cannot be multiplied` — means a layer's `in_features` does not equal the input's last dimension. The cure is not memorizing the message: catch it, print `x.shape`, and size the layer to match.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                       # STEP 1 of the method: make bugs reproducible
device = "cuda" if torch.cuda.is_available() else "cpu"

# A) REPRODUCE A SHAPE MISMATCH, THEN FIX IT
#    Symptom: "mat1 and mat2 shapes cannot be multiplied".
x = torch.randn(16, 20)                     # batch of 16, 20 features each
bad_layer = nn.Linear(10, 4)               # WRONG: expects 10 in-features, gets 20

try:
    bad_layer(x)
except RuntimeError as e:
    print("CAUGHT shape error:", str(e).splitlines()[0])
    # FIX: print the shape, then size the layer to match.
    print("x.shape =", tuple(x.shape))     # -> (16, 20)  => in_features must be 20

good_layer = nn.Linear(20, 4)              # in_features now matches x's last dim
print("fixed output shape:", tuple(good_layer(x).shape))   # -> (16, 4)

### Step 2 — Overfit a single batch (the core sanity check)

The most useful debugging routine: train on ONE fixed batch and see if the loss can reach ~0. If it can, the model and loss are wired correctly and any remaining trouble is in the data, regularization, or learning rate. Print shapes, dtypes, and devices first — the cheapest debug tool — and watch the gradient norm: ~0 means no signal reaching the weights, huge means it is about to explode.

In [ ]:
# B) A TINY MODEL + THE OVERFIT-ONE-BATCH SANITY CHECK
#    If the loss can't reach ~0 on ONE batch, the bug is in
#    the model/loss, not the data or the learning rate.
model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()            # expects LOGITS + LONG class indices

# one fixed batch (note: data must go to the SAME device as the model)
xb = torch.randn(16, 20, device=device)
yb = torch.randint(0, 3, (16,), device=device)   # class indices in [0, 2], dtype long

# Print shapes / dtypes / devices BEFORE the loop — the cheapest debug tool there is.
print("xb:", tuple(xb.shape), xb.dtype, xb.device)
print("yb:", tuple(yb.shape), yb.dtype, yb.device)   # long, same device

for step in range(200):
    optimizer.zero_grad()                  # clear grads — forgetting this is the #1 "loss won't move" bug
    logits = model(xb)                     # raw logits, shape (16, 3) — NO softmax before CrossEntropyLoss
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        # STEP 4: gradient-norm check. ~0 => no signal reaching weights; huge => about to explode.
        total_norm = sum(p.grad.norm().item() ** 2 for p in model.parameters()) ** 0.5
        print(f"step {step:3d}  loss {loss.item():.4f}  grad_norm {total_norm:.3f}")

print("overfit-one-batch final loss:", round(loss.item(), 5), "(should be ~0 -> wiring is correct)")

### Step 3 — Guard against NaN loss and turn on anomaly detection

When the loss goes non-finite, catch it the instant it appears with `torch.isfinite` rather than training through garbage. `set_detect_anomaly(True)` makes autograd point at the exact op that produced the NaN (it is slow, so use it only while hunting). Clipping the gradient norm guards against the exploding-gradient case that often causes the blow-up.

In [ ]:
# C) NaN-LOSS GUARD + ANOMALY DETECTION
#    Catch a NaN the instant it appears and find the op that made it.
torch.autograd.set_detect_anomaly(True)    # slow; turn on only while hunting a NaN / in-place bug

def guarded_step(logits, target):
    loss = loss_fn(logits, target)
    if not torch.isfinite(loss):           # NaN or inf guard
        raise FloatingPointError(f"non-finite loss {loss.item()} — lower lr / clip grads / check inputs")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # guard against exploding grads
    return loss

optimizer.zero_grad()
loss = guarded_step(model(xb), yb)
optimizer.step()
print("guarded step ok, loss:", round(loss.item(), 4))

### Step 4 — Two silent bugs: broadcasting and in-place autograd

Some bugs raise nothing. Subtracting a `(4,)` tensor from a `(4, 1)` tensor *broadcasts* to `(4, 4)` — a silent outer product — so match the shapes deliberately. And editing a leaf tensor in place (`w += 1` when it requires grad) corrupts the backward graph and raises; the fix is to edit weights under `torch.no_grad()` (or out-of-place).

In [ ]:
# D) THE SILENT BROADCASTING BUG (no error, wrong answer)
pred = torch.randn(4, 1)                     # shape (4, 1)
target = torch.randn(4)                      # shape (4,)
wrong = (pred - target)                      # broadcasts to (4, 4) — silently WRONG
right = (pred.squeeze(1) - target)           # shapes match -> (4,)
print("broadcast shapes -> wrong:", tuple(wrong.shape), " right:", tuple(right.shape))

# E) IN-PLACE OP THAT BREAKS AUTOGRAD (and the fix)
w = torch.randn(3, requires_grad=True)
try:
    w += 1                                   # in-place on a leaf that requires grad -> RuntimeError
except RuntimeError as e:
    print("CAUGHT in-place error:", str(e).splitlines()[0])

with torch.no_grad():                        # FIX: edit weights inside no_grad (or use out-of-place w = w + 1)
    w += 1
print("in-place fix ok, w mean:", round(w.mean().item(), 3))

## Visualize the data & results

_Forgetting optimizer.zero_grad() makes gradients accumulate across steps. What does that do to the training-loss curve, compared with the correctly-zeroed run?_

### Step 1 — A tiny regression and its gradient

To see what forgetting `zero_grad()` does, we fit `y = 2x + 1` by hand with plain gradient descent. The `grads` helper returns the mean-squared-error loss and the gradients of the weight and bias — the same quantities PyTorch's autograd would compute for this model.

In [ ]:
import numpy as np

# Tiny linear regression: fit y = 2x + 1 by gradient descent.
rng = np.random.default_rng(0)
N = 64
x = rng.normal(0, 1, N)
y = 2.0 * x + 1.0 + rng.normal(0, 0.1, N)

def grads(w, b):
    err = (w * x + b) - y
    loss = np.mean(err ** 2)
    return loss, 2 * np.mean(err * x), 2 * np.mean(err)

lr, EPOCHS = 0.1, 40

### Step 2 — Run the correct loop vs the buggy one

The FIXED run uses a fresh gradient each step. The BUGGY run accumulates every past gradient and never clears it — exactly what forgetting `optimizer.zero_grad()` does in PyTorch. We record both loss curves so we can compare them.

In [ ]:
# FIXED: fresh gradient each step
w, b, fixed = 0.0, 0.0, []
for _ in range(EPOCHS):
    loss, gw, gb = grads(w, b); fixed.append(loss)
    w -= lr * gw; b -= lr * gb

# BUGGY: gradients accumulate (zero_grad forgotten)
w, b, accw, accb, buggy = 0.0, 0.0, 0.0, 0.0, []
for _ in range(EPOCHS):
    loss, gw, gb = grads(w, b); buggy.append(loss)
    accw += gw; accb += gb            # never cleared
    w -= lr * accw; b -= lr * accb

idx = [0,2,5,7,10,13,15,18,20,23,26,28,31,33,36,39]
print("fixed:", [(i, round(fixed[i], 3)) for i in idx])
print("buggy:", [(i, round(buggy[i], 3)) for i in idx])

### Step 3 — Plot the two training curves

Plotting the two loss curves together makes the bug obvious: the fixed run descends smoothly to ~0, while the accumulating run takes ever-larger effective steps and diverges. That runaway curve is the signature of a forgotten `zero_grad()`.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(buggy, color="#ff7b72", label="buggy (grads accumulate)")
plt.plot(fixed, color="#7ee787", label="fixed (zero_grad each step)")
plt.xlabel("epoch"); plt.ylabel("mean squared error")
plt.title("Buggy (forgot zero_grad) vs fixed training loss")
plt.legend(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Reproduce the classic shape-mismatch error, then read it. Make x = torch.randn(16, 20) and layer = nn.Linear(10, 4), call layer(x) inside a try/except RuntimeError, and print the first line of the message. Then print x.shape and build a correctly-sized layer that works.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap the failing call in try/except RuntimeError and print str(e).splitlines()[0]. — _Catching and reading the message is the first debugging move; the error names the mismatched matrix shapes._
- Print x.shape, then set in_features to the last dim (20). — _nn.Linear needs in_features to equal the input's last dimension — printing the shape reveals the right number._

**Answer:** import torch
import torch.nn as nn
x = torch.randn(16, 20)
bad = nn.Linear(10, 4)
try:
    bad(x)
except RuntimeError as e:
    print(str(e).splitlines()[0])
    # mat1 and mat2 shapes cannot be multiplied (16x20 and 10x4)
print(x.shape)                          # torch.Size([16, 20]) -> in_features must be 20
good = nn.Linear(20, 4)
print(good(x).shape)                    # torch.Size([16, 4])

</details>

**Problem 2.** Type this in Colab. Run the overfit-one-batch sanity check. Build model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3)), a fixed batch xb = torch.randn(16, 20) and yb = torch.randint(0, 3, (16,)) (seed 0), and train on that ONE batch for 200 Adam steps with nn.CrossEntropyLoss. Print the loss every 50 steps. Predict: can it reach ~0?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Loop zero_grad() &rarr; logits = model(xb) &rarr; loss &rarr; backward() &rarr; step() on the same batch. — _A correctly wired model can memorize one batch, driving its loss to ~0 — that proves the wiring is right._
- Predict: yes, it reaches ~0; if it cannot, the bug is in the model or loss. — _The overfit-one-batch test separates wiring bugs from data/regularization/learning-rate issues._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
xb = torch.randn(16, 20)
yb = torch.randint(0, 3, (16,))

for step in range(200):
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward(); opt.step()
    if step % 50 == 0:
        print(step, round(loss.item(), 4))
print("final:", round(loss.item(), 5))   # ~0.0 -> wiring is correct

</details>

**Problem 3.** Type this in Colab. Make NaN loss happen on purpose. Train the same tiny model with an absurd learning rate lr=1e6 for a few steps and print the loss each step until it becomes non-finite. Use torch.isfinite(loss) to detect it and stop.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set a huge lr so the weights blow up, and guard with torch.isfinite(loss). — _Too-high a learning rate is the most common NaN cause; the finite check catches it the instant it appears._
- Break out of the loop on the first non-finite loss. — _Training past a NaN is pointless — the standard triage is to stop, lower the lr, and/or clip gradients._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e6)   # absurdly high
loss_fn = nn.CrossEntropyLoss()
xb = torch.randn(16, 20); yb = torch.randint(0, 3, (16,))

for step in range(20):
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward(); opt.step()
    print(step, loss.item())
    if not torch.isfinite(loss):
        print("non-finite loss -> stop; lower lr / clip grads")  # fires within a few steps
        break

</details>

**Problem 4.** Type this in Colab. Debug a missing gradient. Create w = torch.randn(3, requires_grad=True), then x = w.detach() * 2, then loss = (x ** 2).sum(), call loss.backward() and print w.grad. Predict the result, then fix it so w.grad is not None.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Backprop through a graph that contains .detach() and inspect w.grad. — _detach() cuts the graph, so no gradient flows back to w and w.grad stays None._
- Remove the .detach() so the graph connects loss to w. — _A None grad almost always means the loss does not depend on the parameter through a tracked graph._

**Answer:** w = torch.randn(3, requires_grad=True)
x = w.detach() * 2            # detach CUTS the graph
loss = (x ** 2).sum()
loss.backward()
print(w.grad)                # None  -- no gradient reached w

w = torch.randn(3, requires_grad=True)
x = w * 2                     # FIX: keep w in the graph
loss = (x ** 2).sum()
loss.backward()
print(w.grad)                # tensor([...]) -- real gradients now

</details>

**Problem 5.** Type this in Colab. Reproduce the nn.CrossEntropyLoss double-softmax pitfall as code. With seed 0 and logits = torch.randn(5, 3), y = torch.tensor([0, 2, 1, 0, 1]), compute the loss on (a) the raw logits and (b) logits.softmax(dim=1). Print both. Which is correct?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass raw logits to nn.CrossEntropyLoss — never pre-softmaxed. — _CrossEntropyLoss applies log_softmax internally; feeding probabilities softmaxes twice and shrinks the gradient._
- Compare the two loss values. — _The pre-softmaxed version gives a different, wrong loss — showing why you must feed logits._

**Answer:** torch.manual_seed(0)
loss_fn = nn.CrossEntropyLoss()
logits = torch.randn(5, 3)
y = torch.tensor([0, 2, 1, 0, 1])         # long class indices, shape (5,)

print(round(loss_fn(logits, y).item(), 4))                  # CORRECT (raw logits)
print(round(loss_fn(logits.softmax(dim=1), y).item(), 4))   # WRONG (double softmax)
# The two values differ; only the first uses the intended log_softmax once.

</details>

**Problem 6.** Type this in Colab. Expose the silent broadcasting bug. With pred = torch.randn(4, 1) and target = torch.randn(4) (seed 0), print (pred - target).shape. Predict it BEFORE running. Then fix the shapes two ways and print the corrected shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print the shape of pred - target and predict it first. — _(4,1) broadcasts against (4,) to (4,4) — an outer-product grid, with no error raised._
- Fix with pred.squeeze(1) or target.unsqueeze(1). — _Matching the shapes exactly gives the intended elementwise difference instead of a silent 16-element grid._

**Answer:** torch.manual_seed(0)
pred = torch.randn(4, 1)
target = torch.randn(4)
print((pred - target).shape)              # torch.Size([4, 4])  -- silently WRONG!
print((pred.squeeze(1) - target).shape)   # torch.Size([4])
print((pred - target.unsqueeze(1)).shape) # torch.Size([4, 1])

</details>

**Problem 7.** Type this in Colab. Trigger and fix the in-place autograd error. Make w = torch.randn(3, requires_grad=True) and try w += 1 inside a try/except RuntimeError; print the first line. Then do the same edit safely under torch.no_grad() and print w.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Attempt the in-place w += 1 on a leaf that requires grad and catch the error. — _In-place edits on a leaf tensor autograd is tracking corrupt the backward graph, so PyTorch raises._
- Repeat inside with torch.no_grad(): (or use out-of-place w = w + 1). — _no_grad tells autograd not to track the edit, making manual weight updates legal._

**Answer:** w = torch.randn(3, requires_grad=True)
try:
    w += 1
except RuntimeError as e:
    print(str(e).splitlines()[0])
    # a leaf Variable that requires grad is being used in an in-place operation.
with torch.no_grad():            # FIX: untracked edit
    w += 1
print(w.requires_grad)           # True -- still a leaf that requires grad, edit ok

</details>

**Problem 8.** Type this in Colab. Print shapes, dtypes, and devices through a forward pass — the cheapest debug tool. Build model = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 3)), make xb = torch.randn(16, 20) (seed 0), and print xb.shape, xb.dtype, xb.device, then the logits' shape after the forward.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print .shape, .dtype, .device of the input before the forward. — _Most RuntimeErrors answer themselves once you see the three attributes at the failing line._
- Print the output shape after model(xb). — _Confirming the output shape (16, 3) verifies the layers line up end to end._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 3))
xb = torch.randn(16, 20)
print(xb.shape, xb.dtype, xb.device)   # torch.Size([16, 20]) torch.float32 cpu
logits = model(xb)
print(logits.shape)                    # torch.Size([16, 3])

</details>